# Exploración visual de `immo_scraper_cerdanyola.db`

Notebook aparte para inspeccionar la base de datos local con foco en análisis práctico:
- últimas entradas
- anuncios más antiguos y aún activos
- filtros por precio
- evolución de `price_first_seen` vs `price`
- vista rápida de `first_seen` y `last_seen`

## 1. Conexión y configuración

Abrimos la base SQLite local y dejamos listas algunas utilidades para reutilizar consultas en las siguientes celdas.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display, Markdown

DB_PATH = Path("immo_scraper_cerdanyola.db")
if not DB_PATH.exists():
    raise FileNotFoundError(f"No se encuentra la base de datos: {DB_PATH.resolve()}")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
properties_df = pd.read_sql_query("SELECT * FROM properties", conn)
print(f"Conectado a: {DB_PATH.resolve()}")
print(f"Filas en properties: {len(properties_df)}")

## 2. Resumen rápido

Una panorámica mínima para saber qué datos tenemos antes de filtrar.

In [ ]:
summary = pd.DataFrame({
    'metric': [
        'rows',
        'unique_property_id',
        'active_rows',
        'inactive_rows',
        'price_min',
        'price_median',
        'price_max',
        'price_first_seen_min',
        'price_first_seen_median',
        'price_first_seen_max',
    ],
    'value': [
        len(properties_df),
        properties_df['property_id'].nunique(dropna=True),
        (properties_df['status'] == 'active').sum() if 'status' in properties_df else None,
        (properties_df['status'] == 'inactive').sum() if 'status' in properties_df else None,
        properties_df['price'].min(),
        properties_df['price'].median(),
        properties_df['price'].max(),
        properties_df['price_first_seen'].min() if 'price_first_seen' in properties_df else None,
        properties_df['price_first_seen'].median() if 'price_first_seen' in properties_df else None,
        properties_df['price_first_seen'].max() if 'price_first_seen' in properties_df else None,
    ]
})
display(summary)
display(properties_df[['property_id', 'source', 'title', 'price', 'price_first_seen', 'first_seen', 'last_seen', 'status']].head(10))

## 3. Helpers de inspección

Detectamos automáticamente las columnas temporales y de precio inicial para que la notebook siga funcionando si el esquema cambia un poco.

In [ ]:
properties_columns = pd.read_sql_query("PRAGMA table_info(properties)", conn)['name'].tolist()
first_seen_candidates = ['first_seen', 'first_seen_at', 'created_at', 'published_at', 'date']
last_seen_candidates = ['last_seen', 'last_seen_at', 'updated_at', 'scraped_at', 'date']
price_first_seen_col = 'price_first_seen' if 'price_first_seen' in properties_columns else None
first_seen_col = next((col for col in first_seen_candidates if col in properties_columns), None)
last_seen_col = next((col for col in last_seen_candidates if col in properties_columns), None)

def load_active_properties():
    if 'status' in properties_columns:
        return pd.read_sql_query("SELECT * FROM properties WHERE status = 'active'", conn)
    return properties_df.copy()

def _days_since(values: pd.Series) -> pd.Series:
    now_utc = pd.Timestamp.now(tz='UTC')
    parsed = pd.to_datetime(values, errors='coerce', utc=True)
    return ((now_utc - parsed).dt.total_seconds() // 86400).astype('Int64')

def add_age_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    if first_seen_col and first_seen_col in result.columns:
        result['days_online'] = _days_since(result[first_seen_col])
    else:
        result['days_online'] = pd.Series([pd.NA] * len(result), dtype='Int64')
    if last_seen_col and last_seen_col in result.columns:
        result['days_since_last_seen'] = _days_since(result[last_seen_col])
    else:
        result['days_since_last_seen'] = pd.Series([pd.NA] * len(result), dtype='Int64')
    return result

def compact_cols(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    existing = [col for col in columns if col in df.columns]
    return df[existing] if existing else df

## 4. Últimas entradas

Vista de los anuncios más recientes según `last_seen` o `first_seen`, según lo que exista en la base.

In [ ]:
active_df = add_age_columns(load_active_properties())
sort_col = last_seen_col if last_seen_col in active_df.columns else first_seen_col
sort_col = sort_col if sort_col in active_df.columns else None

recent_columns = [
    'property_id', 'title', 'source', 'city', 'price', 'price_first_seen',
    'first_seen', 'last_seen', 'days_online', 'days_since_last_seen', 'status'
]

if sort_col:
    recent_df = active_df.sort_values(by=sort_col, ascending=False).head(20)
else:
    recent_df = active_df.head(20)

display(compact_cols(recent_df, recent_columns))

## 5. Anuncios más antiguos y aún activos

Ordenados por antigüedad desde `first_seen`, útil para ver qué sigue vivo en la web desde hace más tiempo.

In [ ]:
if first_seen_col and first_seen_col in active_df.columns:
    oldest_df = active_df.sort_values(by=['days_online', 'first_seen'], ascending=[False, True]).head(25)
else:
    oldest_df = active_df.head(25)

display(compact_cols(oldest_df, [
    'property_id', 'title', 'source', 'city', 'price', 'price_first_seen',
    'first_seen', 'last_seen', 'days_online', 'days_since_last_seen', 'status'
]))

## 6. Filtros por precio

Cambia los valores `MIN_PRICE` y `MAX_PRICE` para acotar el rango que quieres inspeccionar.

In [ ]:
MIN_PRICE = 0
MAX_PRICE = 700000
MIN_DAYS_ONLINE = 0

filtered_df = active_df.copy()
if 'price' in filtered_df.columns:
    filtered_df = filtered_df[filtered_df['price'].notna()]
    filtered_df = filtered_df[(filtered_df['price'] >= MIN_PRICE) & (filtered_df['price'] <= MAX_PRICE)]
if 'days_online' in filtered_df.columns:
    filtered_df = filtered_df[filtered_df['days_online'].fillna(-1) >= MIN_DAYS_ONLINE]

display(compact_cols(filtered_df.sort_values(by=['price', 'days_online'], ascending=[True, False]).head(50), [
    'property_id', 'title', 'source', 'city', 'price', 'price_first_seen',
    'first_seen', 'last_seen', 'days_online', 'days_since_last_seen', 'status'
]))

## 7. Evolución de precio

Compara el primer precio visto con el precio actual. Útil para detectar bajadas y subidas.

In [ ]:
if price_first_seen_col and price_first_seen_col in properties_df.columns:
    price_evolution_df = active_df.copy()
    price_evolution_df = price_evolution_df[price_evolution_df[price_first_seen_col].notna() & price_evolution_df['price'].notna()]
    price_evolution_df['price_delta'] = price_evolution_df['price'] - price_evolution_df[price_first_seen_col]
    price_evolution_df['price_delta_pct'] = (price_evolution_df['price_delta'] * 100.0 / price_evolution_df[price_first_seen_col]).round(2)
    display(price_evolution_df.sort_values(by='price_delta', key=lambda s: s.abs(), ascending=False).head(25)[[
        'property_id', 'title', 'source', 'city', 'price_first_seen', 'price', 'price_delta', 'price_delta_pct'
    ]])
else:
    print('No se encontró la columna price_first_seen en properties.')

## 8. Vistas rápidas por tabla

Si quieres explorar una tabla concreta, descomenta una consulta o añade tus propios filtros aquí.

In [ ]:
queries = {
    'active_properties': "SELECT property_id, source, title, price, price_first_seen, first_seen, last_seen, city, status FROM properties WHERE status = 'active' ORDER BY first_seen DESC",
    'oldest_active': "SELECT property_id, source, title, price, price_first_seen, first_seen, last_seen, city, status FROM properties WHERE status = 'active' ORDER BY first_seen ASC",
    'price_drops_or_changes': "SELECT property_id, source, title, price_first_seen, price, first_seen, last_seen FROM properties WHERE price_first_seen IS NOT NULL ORDER BY ABS(price - price_first_seen) DESC",
}

for name, sql in queries.items():
    print(f"\n### {name}")
    display(pd.read_sql_query(sql, conn).head(15))

## Filtrado por precio y evolución global

Estas celdas usan el mismo rango de precio para ver el detalle de los listings y la tendencia global del mercado.

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
from IPython.display import display

DB_PATH = Path("immo_scraper_cerdanyola.db")
MIN_PRICE = 150000
MAX_PRICE = 1000000

with sqlite3.connect(DB_PATH) as conn:
    properties_df = pd.read_sql_query("SELECT * FROM properties", conn)

first_seen_candidates = ["first_seen", "seen_at", "created_at"]
last_seen_candidates = ["last_seen", "updated_at", "seen_at"]
first_seen_col = next((col for col in first_seen_candidates if col in properties_df.columns), None)
last_seen_col = next((col for col in last_seen_candidates if col in properties_df.columns), None)

if first_seen_col is None or last_seen_col is None:
    raise ValueError("No se encontraron columnas temporales compatibles en properties.")

price_window_df = properties_df.copy()
price_window_df["price"] = pd.to_numeric(price_window_df["price"], errors="coerce")
price_window_df[first_seen_col] = pd.to_datetime(price_window_df[first_seen_col], errors="coerce", utc=True)
price_window_df[last_seen_col] = pd.to_datetime(price_window_df[last_seen_col], errors="coerce", utc=True)

now_utc = pd.Timestamp.now(tz="UTC")
price_window_df["days_listed"] = (now_utc - price_window_df[first_seen_col]).dt.days
price_window_df = price_window_df[
    price_window_df["price"].between(MIN_PRICE, MAX_PRICE, inclusive="both")
].copy()

price_window_df = price_window_df.sort_values(
    by=["price", "days_listed", last_seen_col],
    ascending=[True, False, False],
)

cols_to_show = [
    col
    for col in ["source", "title", "price", "days_listed", first_seen_col, last_seen_col, "city", "url"]
    if col in price_window_df.columns
]

display(
    price_window_df[cols_to_show]
    .reset_index(drop=True)
    .head(200)
)

In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

DB_PATH = Path("immo_scraper_cerdanyola.db")
MIN_PRICE = 150000
MAX_PRICE = 1000000

with sqlite3.connect(DB_PATH) as conn:
    properties_lookup = pd.read_sql_query(
        "SELECT property_id, source, title FROM properties",
        conn,
    )
    history_df = pd.read_sql_query(
        "SELECT property_id, price, date FROM price_history",
        conn,
    )

history_df["price"] = pd.to_numeric(history_df["price"], errors="coerce")
history_df["date"] = pd.to_datetime(history_df["date"], errors="coerce", utc=True)
history_df = history_df.dropna(subset=["property_id", "price", "date"])

history_df = history_df.merge(properties_lookup, on="property_id", how="inner")
history_df = history_df[history_df["price"].between(MIN_PRICE, MAX_PRICE, inclusive="both")].copy()

history_df["day"] = history_df["date"].dt.floor("D")
daily_price_df = (
    history_df.groupby("day", as_index=False)
    .agg(
        median_price=("price", "median"),
        mean_price=("price", "mean"),
        listings=("property_id", "nunique"),
    )
    .sort_values("day")
)

daily_price_df["median_price_rolling_7d"] = daily_price_df["median_price"].rolling(7, min_periods=1).mean()
daily_price_df["mean_price_rolling_7d"] = daily_price_df["mean_price"].rolling(7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(daily_price_df["day"], daily_price_df["median_price"], alpha=0.25, linewidth=1.5, label="Mediana diaria")
ax.plot(daily_price_df["day"], daily_price_df["median_price_rolling_7d"], linewidth=2.8, label="Mediana móvil 7 días")
ax.set_title(f"Evolución global de precios entre {MIN_PRICE}€ y {MAX_PRICE}€")
ax.set_xlabel("Fecha")
ax.set_ylabel("Precio (€)")
ax.grid(True, alpha=0.25)
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

display(daily_price_df.tail(20))

## Análisis de deltas de precio por propiedad individual

Ver cuáles están bajando, cuánto y desde cuándo se listan (sin promediar por tipo).

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
from IPython.display import display

DB_PATH = Path("immo_scraper_cerdanyola.db")
MIN_PRICE = 150000
MAX_PRICE = 1000000

with sqlite3.connect(DB_PATH) as conn:
    properties_df = pd.read_sql_query("SELECT * FROM properties", conn)
    price_history_all = pd.read_sql_query(
        """
        SELECT ph.property_id, ph.price, ph.date, p.source, p.title
        FROM price_history ph
        JOIN properties p ON p.property_id = ph.property_id
        ORDER BY ph.property_id, ph.date
        """,
        conn,
    )

price_history_all["price"] = pd.to_numeric(price_history_all["price"], errors="coerce")
price_history_all["date"] = pd.to_datetime(price_history_all["date"], errors="coerce", utc=True)
price_history_all = price_history_all.dropna(subset=["property_id", "price", "date"])

first_price = price_history_all.groupby("property_id").first().reset_index()[["property_id", "price"]]
first_price.rename(columns={"price": "price_initial"}, inplace=True)

last_price = price_history_all.groupby("property_id").last().reset_index()[["property_id", "price", "date"]]
last_price.rename(columns={"price": "price_current", "date": "last_price_date"}, inplace=True)

price_changes = first_price.merge(last_price, on="property_id", how="inner")
price_changes = price_changes.merge(
    properties_df[["property_id", "source", "title", "first_seen", "last_seen"]],
    on="property_id",
    how="inner",
)

price_changes["price_current"] = pd.to_numeric(price_changes["price_current"], errors="coerce")
price_changes["price_current"] = price_changes["price_current"].fillna(price_changes["price_initial"])
price_changes["first_seen"] = pd.to_datetime(price_changes["first_seen"], errors="coerce", utc=True)
price_changes["last_seen"] = pd.to_datetime(price_changes["last_seen"], errors="coerce", utc=True)

price_changes = price_changes[
    price_changes["price_current"].between(MIN_PRICE, MAX_PRICE, inclusive="both")
].copy()

price_changes["delta_abs"] = price_changes["price_current"] - price_changes["price_initial"]
price_changes["delta_pct"] = (price_changes["delta_abs"] / price_changes["price_initial"] * 100).round(2)

now_utc = pd.Timestamp.now(tz="UTC")
price_changes["days_listed"] = (now_utc - price_changes["first_seen"]).dt.days

price_changes = price_changes.sort_values(by="delta_pct", ascending=True)

cols_display = ["source", "title", "price_initial", "price_current", "delta_abs", "delta_pct", "days_listed", "first_seen"]
display(
    price_changes[cols_display]
    .reset_index(drop=True)
    .head(150)
)

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

DB_PATH = Path("immo_scraper_cerdanyola.db")
MIN_PRICE = 150000
MAX_PRICE = 1000000

with sqlite3.connect(DB_PATH) as conn:
    properties_df = pd.read_sql_query("SELECT * FROM properties", conn)
    price_history_all = pd.read_sql_query(
        """
        SELECT ph.property_id, ph.price, ph.date, p.source, p.title
        FROM price_history ph
        JOIN properties p ON p.property_id = ph.property_id
        ORDER BY ph.property_id, ph.date
        """,
        conn,
    )

price_history_all["price"] = pd.to_numeric(price_history_all["price"], errors="coerce")
price_history_all["date"] = pd.to_datetime(price_history_all["date"], errors="coerce", utc=True)
price_history_all = price_history_all.dropna(subset=["property_id", "price", "date"])

first_price = price_history_all.groupby("property_id").first().reset_index()[["property_id", "price"]]
first_price.rename(columns={"price": "price_initial"}, inplace=True)

last_price = price_history_all.groupby("property_id").last().reset_index()[["property_id", "price"]]
last_price.rename(columns={"price": "price_current"}, inplace=True)

price_changes = first_price.merge(last_price, on="property_id", how="inner")
price_changes = price_changes.merge(
    properties_df[["property_id", "source", "title", "first_seen"]],
    on="property_id",
    how="inner",
)

price_changes["price_current"] = pd.to_numeric(price_changes["price_current"], errors="coerce")
price_changes["price_current"] = price_changes["price_current"].fillna(price_changes["price_initial"])
price_changes["first_seen"] = pd.to_datetime(price_changes["first_seen"], errors="coerce", utc=True)

price_changes = price_changes[
    price_changes["price_current"].between(MIN_PRICE, MAX_PRICE, inclusive="both")
].copy()

price_changes["delta_abs"] = price_changes["price_current"] - price_changes["price_initial"]
price_changes["delta_pct"] = (price_changes["delta_abs"] / price_changes["price_initial"] * 100).round(2)

now_utc = pd.Timestamp.now(tz="UTC")
price_changes["days_listed"] = (now_utc - price_changes["first_seen"]).dt.days

# Preparar formato de precios para hover
price_changes["price_initial_fmt"] = price_changes["price_initial"].apply(lambda x: f"€{x:,.0f}")
price_changes["price_current_fmt"] = price_changes["price_current"].apply(lambda x: f"€{x:,.0f}")

# Crear subplots: histograma + scatter
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Distribución de deltas ({MIN_PRICE}€-{MAX_PRICE}€)",
        "Deltas vs tiempo de listado"
    )
)

# Histograma con plotly
hist_data = px.histogram(
    price_changes,
    x="delta_pct",
    nbins=30,
    title="Histograma",
    color_discrete_sequence=["steelblue"]
).data[0]

fig.add_trace(hist_data, row=1, col=1)

# Scatter con hover info detallada
scatter = px.scatter(
    price_changes,
    x="days_listed",
    y="delta_pct",
    color="delta_pct",
    color_continuous_scale="RdYlGn",
    hover_data={
        "title": True,
        "source": True,
        "price_initial_fmt": ":.0",
        "price_current_fmt": ":.0",
        "delta_pct": ":.2f",
        "delta_abs": ":.0f",
        "days_listed": ":.0f",
        "delta_pct": True,  # repetir para mostrar en hover
    },
    custom_data=["title", "source", "price_initial_fmt", "price_current_fmt", "delta_abs"],
).data[0]

fig.add_trace(scatter, row=1, col=2)

# Actualizar layout del scatter
fig.add_hline(y=0, line_dash="dash", line_color="black", row=1, col=2)

# Personalizar ejes
fig.update_xaxes(title_text="Cambio porcentual (%)", row=1, col=1)
fig.update_yaxes(title_text="Cantidad de anuncios", row=1, col=1)

fig.update_xaxes(title_text="Días listados", row=1, col=2)
fig.update_yaxes(title_text="Cambio porcentual (%)", row=1, col=2)

# Mejorar hover template para scatter
hover_template = (
    "<b>%{customdata[1]}</b><br>"
    "Título: %{customdata[0]}<br>"
    "Precio inicial: %{customdata[2]}<br>"
    "Precio actual: %{customdata[3]}<br>"
    "Cambio: %{y:.2f}% (€%{customdata[4]:,.0f})<br>"
    "Días listados: %{x}<br>"
    "<extra></extra>"
)

fig.data[1].hovertemplate = hover_template

fig.update_layout(
    height=500,
    width=1200,
    showlegend=False,
    title_text="Análisis de Deltas de Precio (Interactivo con Plotly)",
    coloraxis_colorbar=dict(title="% Cambio"),
)

fig.show()

# Tabla de estadísticas
stats = {
    "Total de anuncios": len(price_changes),
    "Con bajada de precio": (price_changes["delta_pct"] < 0).sum(),
    "Con subida de precio": (price_changes["delta_pct"] > 0).sum(),
    "Sin cambio": (price_changes["delta_pct"] == 0).sum(),
    "Mediana de cambio (%)": price_changes["delta_pct"].median(),
    "Media de cambio (%)": price_changes["delta_pct"].mean(),
    "Max bajada (%)": price_changes["delta_pct"].min(),
    "Max subida (%)": price_changes["delta_pct"].max(),
}

display(pd.DataFrame(stats, index=[0]).T.rename(columns={0: "Valor"}))

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

DB_PATH = Path("immo_scraper_cerdanyola.db")

with sqlite3.connect(DB_PATH) as conn:
    props = pd.read_sql_query(
        "SELECT property_id, source, title, price_first_seen, price, first_seen, last_seen, status FROM properties",
        conn,
    )

# Parse datetimes
props["first_seen"] = pd.to_datetime(props["first_seen"], errors="coerce", utc=True)
props["last_seen"] = pd.to_datetime(props["last_seen"], errors="coerce", utc=True)

# Normalize to dates for daily grouping
props["date_first"] = props["first_seen"].dt.normalize()
props["date_last"] = props["last_seen"].dt.normalize()

# New entries per day
new_by_day = props.groupby("date_first").size().rename("new")
# Removed per day: use rows that are inactive (mark_inactive sets status='inactive' and updates last_seen)
removed_by_day = (
    props[props["status"] == "inactive"].groupby("date_last").size().rename("removed")
)

# Build full date index
min_date = min(props["date_first"].min(), props["date_last"].min())
max_date = max(props["date_first"].max(), props["date_last"].max())
if pd.isna(min_date):
    min_date = pd.Timestamp.now(tz="UTC").normalize()
if pd.isna(max_date):
    max_date = min_date

date_index = pd.date_range(start=min_date, end=max_date, freq="D", tz="UTC")

df_days = pd.DataFrame(index=date_index)

df_days = df_days.join(new_by_day.reindex(date_index, fill_value=0))

df_days = df_days.join(removed_by_day.reindex(date_index, fill_value=0))

# Cumulative active (approx): cumulative new minus cumulative removed
df_days["cumulative_new"] = df_days["new"].cumsum()
df_days["cumulative_removed"] = df_days["removed"].cumsum()
df_days["active_estimated"] = df_days["cumulative_new"] - df_days["cumulative_removed"]

# Prepare figure: stacked bars (new vs removed) + line active
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Bars: new (green) and removed (red)
fig.add_trace(
    go.Bar(x=df_days.index, y=df_days["new"], name="Entradas (nuevas)", marker_color="green", hovertemplate="%{x|%Y-%m-%d}: %{y} nuevas<extra></extra>"),
    secondary_y=False,
)
fig.add_trace(
    go.Bar(x=df_days.index, y=df_days["removed"], name="Salidas (eliminadas)", marker_color="red", hovertemplate="%{x|%Y-%m-%d}: %{y} salidas<extra></extra>"),
    secondary_y=False,
)

# Line: estimated active
fig.add_trace(
    go.Scatter(x=df_days.index, y=df_days["active_estimated"], mode="lines+markers", name="Activas estimadas", line=dict(color="blue"), hovertemplate="%{x|%Y-%m-%d}: %{y} activas<extra></extra>"),
    secondary_y=True,
)

fig.update_layout(barmode="stack", title_text="Entradas vs Salidas diarias y total activo estimado", height=500, width=1000)
fig.update_xaxes(title_text="Fecha")
fig.update_yaxes(title_text="Entradas / Salidas", secondary_y=False)
fig.update_yaxes(title_text="Activas estimadas", secondary_y=True)

fig.show()

# Mostrar propiedades recientemente eliminadas y su histórico
recent_removed = props[props["status"] == "inactive"].sort_values("last_seen", ascending=False)
cols = ["property_id", "source", "title", "price_first_seen", "price", "first_seen", "last_seen"]
display(recent_removed[cols].head(30).reset_index(drop=True))

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
from IPython.display import display, Markdown

DB_PATH = Path("immo_scraper_cerdanyola.db")
LOOKBACK_DAYS = 30

with sqlite3.connect(DB_PATH) as conn:
    properties_cols = pd.read_sql_query("PRAGMA table_info(properties)", conn)["name"].tolist()

    required = {"property_id", "status", "last_seen", "price"}
    if not required.issubset(set(properties_cols)):
        missing = sorted(required - set(properties_cols))
        raise ValueError(f"Faltan columnas en properties para este análisis: {missing}")

    select_cols = [
        c
        for c in [
            "property_id",
            "source",
            "title",
            "city",
            "status",
            "first_seen",
            "last_seen",
            "price_first_seen",
            "price",
            "url",
        ]
        if c in properties_cols
    ]

    removed_df = pd.read_sql_query(
        f"SELECT {', '.join(select_cols)} FROM properties WHERE status = 'inactive'",
        conn,
    )

removed_df["last_seen"] = pd.to_datetime(removed_df["last_seen"], errors="coerce", utc=True)
if "first_seen" in removed_df.columns:
    removed_df["first_seen"] = pd.to_datetime(removed_df["first_seen"], errors="coerce", utc=True)
if "price_first_seen" in removed_df.columns:
    removed_df["price_first_seen"] = pd.to_numeric(removed_df["price_first_seen"], errors="coerce")
if "price" in removed_df.columns:
    removed_df["price"] = pd.to_numeric(removed_df["price"], errors="coerce")

now_utc = pd.Timestamp.now(tz="UTC")
cutoff = now_utc - pd.Timedelta(days=LOOKBACK_DAYS)

removed_last_month = removed_df[removed_df["last_seen"].notna() & (removed_df["last_seen"] >= cutoff)].copy()

if removed_last_month.empty:
    display(Markdown(f"### No hay anuncios marcados como inactive en los últimos {LOOKBACK_DAYS} días."))
else:
    # Reconstruimos el precio de salida con el último precio en price_history <= last_seen
    with sqlite3.connect(DB_PATH) as conn:
        tables = set(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)["name"].tolist())

        removed_last_month["property_id"] = removed_last_month["property_id"].astype(str)
        removed_last_month = removed_last_month.rename(columns={"price": "price_in_properties"})
        removed_last_month = removed_last_month.reset_index(drop=True)
        removed_last_month["row_id"] = removed_last_month.index

        if "price_history" in tables:
            property_ids = removed_last_month["property_id"].dropna().unique().tolist()

            if property_ids:
                placeholders = ",".join(["?"] * len(property_ids))
                history_df = pd.read_sql_query(
                    f"SELECT property_id, price, date FROM price_history WHERE property_id IN ({placeholders})",
                    conn,
                    params=property_ids,
                )
            else:
                history_df = pd.DataFrame(columns=["property_id", "price", "date"])

            if not history_df.empty:
                history_df["property_id"] = history_df["property_id"].astype(str)
                history_df["price"] = pd.to_numeric(history_df["price"], errors="coerce")
                history_df["date"] = pd.to_datetime(history_df["date"], errors="coerce", utc=True)
                history_df = history_df.dropna(subset=["property_id", "price", "date"])

                candidates = removed_last_month[["row_id", "property_id", "last_seen"]].merge(
                    history_df,
                    on="property_id",
                    how="left",
                )
                candidates = candidates[candidates["date"].notna() & (candidates["date"] <= candidates["last_seen"])].copy()

                if not candidates.empty:
                    last_idx = candidates.groupby("row_id")["date"].idxmax()
                    last_prices = candidates.loc[last_idx, ["row_id", "price", "date"]].rename(
                        columns={"price": "price_at_exit", "date": "price_at_exit_date"}
                    )
                    merged = removed_last_month.merge(last_prices, on="row_id", how="left")
                else:
                    merged = removed_last_month.copy()
                    merged["price_at_exit"] = pd.NA
                    merged["price_at_exit_date"] = pd.NaT
            else:
                merged = removed_last_month.copy()
                merged["price_at_exit"] = pd.NA
                merged["price_at_exit_date"] = pd.NaT
        else:
            merged = removed_last_month.copy()
            merged["price_at_exit"] = pd.NA
            merged["price_at_exit_date"] = pd.NaT

    merged["price_at_exit"] = pd.to_numeric(merged["price_at_exit"], errors="coerce")
    merged["price_at_exit"] = merged["price_at_exit"].fillna(merged["price_in_properties"])

    if "price_first_seen" in merged.columns:
        merged["exit_vs_first_abs"] = merged["price_at_exit"] - merged["price_first_seen"]
        merged["exit_vs_first_pct"] = (merged["exit_vs_first_abs"] * 100.0 / merged["price_first_seen"]).round(2)

    summary = {
        "Periodo analizado": f"Ultimos {LOOKBACK_DAYS} dias",
        "Fecha corte (UTC)": cutoff.strftime("%Y-%m-%d %H:%M:%S"),
        "Anuncios retirados": int(len(merged)),
        "Con precio de salida": int(merged["price_at_exit"].notna().sum()),
        "Precio salida mediano": float(merged["price_at_exit"].median()) if merged["price_at_exit"].notna().any() else None,
        "Precio salida medio": float(merged["price_at_exit"].mean()) if merged["price_at_exit"].notna().any() else None,
    }

    if "exit_vs_first_pct" in merged.columns and merged["exit_vs_first_pct"].notna().any():
        summary["% mediano vs primer precio"] = float(merged["exit_vs_first_pct"].median())
        summary["% medio vs primer precio"] = float(merged["exit_vs_first_pct"].mean())

    display(Markdown("### Salidas del ultimo mes: precio al retirar"))
    display(pd.DataFrame(summary, index=[0]).T.rename(columns={0: "valor"}))

    detail_cols = [
        c
        for c in [
            "property_id",
            "source",
            "title",
            "city",
            "price_first_seen",
            "price_in_properties",
            "price_at_exit",
            "exit_vs_first_abs",
            "exit_vs_first_pct",
            "price_at_exit_date",
            "first_seen",
            "last_seen",
            "url",
        ]
        if c in merged.columns
    ]

    detail_df = merged.sort_values("last_seen", ascending=False).reset_index(drop=True)
    display(detail_df[detail_df["price_at_exit"] <= 700000][detail_cols])

## 9. Evolución temporal de salidas y precio mediano

Serie temporal de anuncios retirados por periodo y su precio mediano estimado en el momento de salida.

- `LOOKBACK_DAYS`: ventana temporal a analizar.
- `FREQ`: granularidad (`"D"`, `"W"`, `"M"`).

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

DB_PATH = Path("immo_scraper_cerdanyola.db")
LOOKBACK_DAYS = 180
FREQ = "D"  # "D" (dia), "W" (semana), "M" (mes)

with sqlite3.connect(DB_PATH) as conn:
    props_cols = pd.read_sql_query("PRAGMA table_info(properties)", conn)["name"].tolist()

    required = {"property_id", "status", "last_seen", "price"}
    if not required.issubset(set(props_cols)):
        missing = sorted(required - set(props_cols))
        raise ValueError(f"Faltan columnas en properties para este análisis: {missing}")

    select_cols = [
        c
        for c in ["property_id", "source", "title", "last_seen", "price"]
        if c in props_cols
    ]

    removed_df = pd.read_sql_query(
        f"SELECT {', '.join(select_cols)} FROM properties WHERE status = 'inactive'",
        conn,
    )

removed_df["property_id"] = removed_df["property_id"].astype(str)
removed_df["last_seen"] = pd.to_datetime(removed_df["last_seen"], errors="coerce", utc=True)
removed_df["price"] = pd.to_numeric(removed_df["price"], errors="coerce")

cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=LOOKBACK_DAYS)
removed_df = removed_df[removed_df["last_seen"].notna() & (removed_df["last_seen"] >= cutoff)].copy()

if removed_df.empty:
    display(Markdown(f"### No hay salidas en los últimos {LOOKBACK_DAYS} días."))
else:
    removed_df = removed_df.reset_index(drop=True)
    removed_df["row_id"] = removed_df.index
    removed_df = removed_df.rename(columns={"price": "price_in_properties"})

    with sqlite3.connect(DB_PATH) as conn:
        table_names = set(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)["name"].tolist())

        if "price_history" in table_names:
            property_ids = removed_df["property_id"].dropna().unique().tolist()
            if property_ids:
                placeholders = ",".join(["?"] * len(property_ids))
                history_df = pd.read_sql_query(
                    f"SELECT property_id, price, date FROM price_history WHERE property_id IN ({placeholders})",
                    conn,
                    params=property_ids,
                )
            else:
                history_df = pd.DataFrame(columns=["property_id", "price", "date"])
        else:
            history_df = pd.DataFrame(columns=["property_id", "price", "date"])

    if not history_df.empty:
        history_df["property_id"] = history_df["property_id"].astype(str)
        history_df["price"] = pd.to_numeric(history_df["price"], errors="coerce")
        history_df["date"] = pd.to_datetime(history_df["date"], errors="coerce", utc=True)
        history_df = history_df.dropna(subset=["property_id", "price", "date"])

        candidates = removed_df[["row_id", "property_id", "last_seen"]].merge(
            history_df,
            on="property_id",
            how="left",
        )
        candidates = candidates[candidates["date"].notna() & (candidates["date"] <= candidates["last_seen"])].copy()

        if not candidates.empty:
            idx = candidates.groupby("row_id")["date"].idxmax()
            exit_prices = candidates.loc[idx, ["row_id", "price"]].rename(columns={"price": "price_at_exit"})
            removed_df = removed_df.merge(exit_prices, on="row_id", how="left")
        else:
            removed_df["price_at_exit"] = pd.NA
    else:
        removed_df["price_at_exit"] = pd.NA

    removed_df["price_at_exit"] = pd.to_numeric(removed_df["price_at_exit"], errors="coerce")
    removed_df["price_at_exit"] = removed_df["price_at_exit"].fillna(removed_df["price_in_properties"])

    # Bucket temporal (dia/semana/mes) a partir de la fecha de salida.
    last_seen_naive = removed_df["last_seen"].dt.tz_convert(None)
    removed_df["period"] = last_seen_naive.dt.to_period(FREQ).dt.to_timestamp().dt.tz_localize("UTC")

    evolution_df = (
        removed_df.groupby("period", as_index=False)
        .agg(
            n_salidas=("row_id", "count"),
            precio_mediano_salida=("price_at_exit", "median"),
            precio_medio_salida=("price_at_exit", "mean"),
        )
        .sort_values("period")
    )

    evolution_df["n_salidas_roll_4"] = evolution_df["n_salidas"].rolling(4, min_periods=1).mean()
    evolution_df["precio_mediano_roll_4"] = evolution_df["precio_mediano_salida"].rolling(4, min_periods=1).mean()

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(
        go.Bar(
            x=evolution_df["period"],
            y=evolution_df["n_salidas"],
            name="Salidas",
            marker_color="#d1495b",
            hovertemplate="%{x|%Y-%m-%d}: %{y} salidas<extra></extra>",
        ),
        secondary_y=False,
    )

    fig.add_trace(
        go.Scatter(
            x=evolution_df["period"],
            y=evolution_df["n_salidas_roll_4"],
            mode="lines",
            name="Salidas (media movil 4 periodos)",
            line=dict(color="#8c1c13", width=2),
            hovertemplate="%{x|%Y-%m-%d}: %{y:.2f} salidas (MM4)<extra></extra>",
        ),
        secondary_y=False,
    )

    fig.add_trace(
        go.Scatter(
            x=evolution_df["period"],
            y=evolution_df["precio_mediano_salida"],
            mode="lines+markers",
            name="Precio mediano salida",
            line=dict(color="#2f6690", width=2.5),
            marker=dict(size=6),
            hovertemplate="%{x|%Y-%m-%d}: %{y:,.0f} EUR<extra></extra>",
        ),
        secondary_y=True,
    )

    fig.add_trace(
        go.Scatter(
            x=evolution_df["period"],
            y=evolution_df["precio_mediano_roll_4"],
            mode="lines",
            name="Precio mediano (media movil 4 periodos)",
            line=dict(color="#16425b", width=2, dash="dash"),
            hovertemplate="%{x|%Y-%m-%d}: %{y:,.0f} EUR (MM4)<extra></extra>",
        ),
        secondary_y=True,
    )

    fig.update_layout(
        title=f"Evolución diaria de salidas y precio mediano de salida (ultimos {LOOKBACK_DAYS} dias)",
        barmode="group",
        height=520,
        width=1100,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )
    fig.update_xaxes(title_text="Dia de salida")
    fig.update_yaxes(title_text="Numero de salidas", secondary_y=False)
    fig.update_yaxes(title_text="Precio mediano salida (EUR)", secondary_y=True)

    fig.show()

    summary = {
        "Periodos analizados": int(len(evolution_df)),
        "Salidas totales": int(evolution_df["n_salidas"].sum()),
        "Salidas medianas por periodo": float(evolution_df["n_salidas"].median()),
        "Precio mediano global de salida": float(removed_df["price_at_exit"].median()) if removed_df["price_at_exit"].notna().any() else None,
    }
    display(pd.DataFrame(summary, index=[0]).T.rename(columns={0: "valor"}))

    display(evolution_df.tail(20).reset_index(drop=True))